### DSCI 410L Project data cleaning

# 1 CLEANING
- load data files
- subsection dataframes to necessary columns only 
- rename/update values in columns to be consistant

- Do the above steps for MCSLC, Eugene, and Springfield data then combine MCSLC with the Eugene and Springfile data finishing with dataframes "eugene" and "springfield".

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# MCSLC

In [ ]:
#read file
#mcslc = pd.read_csv("MCSLC.csv")
mcslc = pd.read_excel("MCSLC.xlsx")
#subsection the DF to only necessary columns
mcslc_clean = mcslc[["ID", "City", "Dispatch Date & Time", "Disposition"]].dropna()

#Clean closed_as column
mcslc_clean = mcslc_clean.rename(columns={"Disposition": "closed_as"}) #renaming the disposition column for consistancy between dataframes
mcslc_clean["closed_as"] = mcslc_clean["closed_as"].str.upper()
#change emergency department labal to HOSPITAL
mcslc_clean.replace('EMERGENCY DEPARTMENT', 'HOSPITAL', inplace=True)


#Clean City Column
mcslc_clean.drop(mcslc_clean[mcslc_clean["City"].isin(["Unknown", "Other", "Out of County"])].index, inplace=True)

#Change the datetime column to actual datetime variables
mcslc_clean["calltime"]= pd.to_datetime(mcslc_clean["Dispatch Date & Time"], format="%m/%d/%y %H:%M")

#Agency column
mcslc_clean["agency"] = "MCSLC"

mcslc_eug = mcslc_clean.where(mcslc_clean["City"] == "Eugene").dropna()
mcslc_spd = mcslc_clean.where(mcslc_clean["City"] == "Springfield").dropna()

mcslc_spd.head()

,ID,City,Dispatch Date & Time,closed_as,calltime,agency
33,14472.0,Springfield,11/4/25 5:42,HOSPITAL,2025-11-04 05:42:00,MCSLC
39,12245.0,Springfield,8/22/24 0:00,HOSPITAL,2024-08-22 00:00:00,MCSLC
48,10052.0,Springfield,1/14/25 15:04,HOSPITAL,2025-01-14 15:04:00,MCSLC
49,10062.0,Springfield,1/16/25 15:19,HOSPITAL,2025-01-16 15:19:00,MCSLC
51,10070.0,Springfield,2/1/25 16:31,HOSPITAL,2025-02-01 16:31:00,MCSLC


# Eugene


In [413]:
years = list(range(2015, 2026))

# import EUG CAD data
cad_list = []
for year in years:
    df = pd.read_csv(f"Eugene_CAD_data_noloc/EugeneCAD{year}noloc.csv", low_memory=False)
    cad_list.append(df)

eug = pd.concat(cad_list, ignore_index=True)
eug.head(3)

,yr,agency,service,inci_id,calltime,case_id,callsource,nature,closecode,closed_as,secs_to_disp,secs_to_arrv,secs_to_close,disp,arrv,priority,primeunit,units_dispd,units_arrived,month
0,2015,EPD,LAW,15000001,2015-01-01 00:00:00.000,NaN,SELF,PERSON STOP,ASST,ASSISTED,0.0,0.0,217,1,1,6,_5E48,1,1,NaN
1,2015,EPD,LAW,15000002,2015-01-01 00:00:44.000,NaN,SELF,FIGHT,RSLV,RESOLVED,0.0,0.0,2114,1,1,P,_3F65,4,2,NaN
2,2015,EPD,LAW,15000003,2015-01-01 00:01:05.000,NaN,PHONE,CHECK WELFARE,ASST,ASSISTED,708.0,1094.0,1538,1,1,5,_3J79,1,1,NaN


In [227]:
eug_clean = eug[["calltime", "closed_as", "agency", "primeunit"]]
eug_clean = eug_clean.dropna()

#closed_as column
#change to K DEPLOYED maybe K9 units?
eug_clean.replace(['K897 DEPLOYED', 'K891 DEPLOYED', 'K895 DEPLOYED', 'K898 DEPLOYED', 'K896 DEPLOYED', 'K893 DEPLOYED'], "K DEPLOYED", inplace=True)
#assumption this is hospital transport: 'PATIENT TRANSPORTED' and changed to 'HOSPITAL'
eug_clean.replace(["PATIENT TRANSPORTED", 'MEDICAL AID EPD', 'SEND PCR REPORT'], "HOSPITAL", inplace=True)
#should 'JUVENILE TAKEN INTO CUSTODY' count as 'ARREST'?
eug_clean.replace("JUVENILE TAKEN INTO CUSTODY", "ARREST", inplace=True)


#change calltime to datetime objects
eug_clean["calltime"] = pd.to_datetime(eug_clean["calltime"], format="%Y-%m-%d %H:%M:%S.%f")


#Make is_CAHOOTS column
#True if agency==CAHE and/or primeunit follows the pattern _#J##
eug_clean["is_cahoots"] = (eug_clean["agency"] == "CAHE" ) | (eug_clean["primeunit"].isin(['_TESTCA', '_CAHOT ', '_1J77  ', '_3J78  ', '_4J79  ','_AWO   ', '_C100  ', '_3J77  ', '_3J79  ']))
eug_clean.loc[eug_clean["is_cahoots"] == True, "agency"] = "CAHE"
eug_clean.head()

,calltime,closed_as,agency,primeunit,is_cahoots
0,2015-01-01 00:00:00,ASSISTED,EPD,_5E48,False
1,2015-01-01 00:00:44,RESOLVED,EPD,_3F65,False
2,2015-01-01 00:01:05,ASSISTED,CAHE,_3J79,True
3,2015-01-01 00:03:16,PATROL CHECK,EPD,_5E48,False
4,2015-01-01 00:03:34,ADVISED,EPD,_5K97,False


In [ ]:
#Merge Eug and MCSLC
mcslc_eug["is_cahoots"] = False
mcslc_eug.drop(["City", "ID", "Dispatch Date & Time"], axis=1, inplace=True)

eugene = pd.concat([eug_clean, mcslc_eug], ignore_index=True).sort_values(by="calltime")
eugene["yr"] = eugene["calltime"].dt.year
eugene["month"] = eugene["calltime"].dt.month
eugene.head(3)

,calltime,closed_as,agency,primeunit,is_cahoots,yr,month
0,2015-01-01 00:00:00,ASSISTED,EPD,_5E48,False,2015,1
1,2015-01-01 00:00:44,RESOLVED,EPD,_3F65,False,2015,1
2,2015-01-01 00:01:05,ASSISTED,CAHE,_3J79,True,2015,1


# Springfield

In [395]:
#Read in all files
service_files = glob.glob("C:/Users/brook/OneDrive/Desktop/DSCI 410L/2015-2025 Calls for Service/*.csv")
spd = pd.concat((pd.read_csv(f) for f in service_files), ignore_index=True)

units_files = glob.glob("C:/Users/brook/OneDrive/Desktop/DSCI 410L/2015-2025 Responding Units/*.csv")
spd_units = pd.concat((pd.read_csv(f) for f in units_files), ignore_index=True)

close_files = glob.glob("C:/Users/brook/OneDrive/Desktop/DSCI 410L/2015-2025 SPD Calls with Close Codes/*.csv")
spd_close = pd.concat((pd.read_csv(f) for f in close_files), ignore_index=True)

designator = pd.read_csv("SPD Designator Notes.csv")
close_codes = pd.read_csv("SPD Close Codes.csv")

In [396]:
#Merge dataframes
spd_new = spd.merge(spd_close, on="Incident Number", how="outer")
spd_new = spd_new.merge(spd_units, left_on="Incident Number", right_on="inci id", how="outer")

spd_clean = spd_new[['Responding Agency', 'Close Code', 'call time', 'prime unit', 'Call Year']].copy()
spd_clean.rename(columns={"Responding Agency": "agency"}, inplace=True)

mapping = dict(zip(close_codes["Close Code"], close_codes["Definition"]))
spd_clean["closed_as"] = (spd_clean["Close Code"].map(mapping))
spd_clean["closed_as"] = spd_clean["closed_as"].str.upper()

#convert to datetime
#spd_clean["calltime"] = pd.to_datetime(spd_clean["call time"], format="%m/%d/%Y %H:%M")
spd_clean["calltime"] = pd.to_datetime(spd_clean["call time"], errors="coerce")
spd_clean["yr"] = spd_clean["calltime"].dt.year
spd_clean["month"] = spd_clean["calltime"].dt.month
spd_clean.drop(["Close Code", "call time"], axis=1, inplace=True)

#create is_cahoots boolean column
spd_clean["is_cahoots"] = spd_clean["prime unit"].isin(['3J81  ', 'CAHOT ', '3J78  '])
spd_clean.loc[spd_clean["is_cahoots"] == True, "agency"] = "CAHE"


#updating closed as variables similar to eugene
spd_clean.replace(['MEDICAL AID EPD', 'MED EXPRESS', 'PATIENT TRASNPORTED', 'SEND PCR REPORT'], "HOSPITAL", inplace=True)
spd_clean.replace("JUVENILE TAKEN INTO CUSTODY", "ARREST", inplace=True)

spd_clean.head(5)


,agency,prime unit,Call Year,closed_as,calltime,yr,month,is_cahoots
0,SPD,1S12,2015.0,ASSISTED,2015-01-01 00:01:00,2015,1,False
1,SPD,,2015.0,DISREGARD,2015-01-01 00:04:00,2015,1,False
2,SPD,3S12,2015.0,ARREST,2015-01-01 00:11:00,2015,1,False
3,SPD,1S12,2015.0,TRAFFIC STOP ONLY,2015-01-01 00:14:00,2015,1,False
4,SPD,1S18,2015.0,TRAFFIC STOP ONLY,2015-01-01 00:17:00,2015,1,False


In [ ]:
#Merge MCSLC and Springfield
#mcslc_spd.drop(["City", "Dispatch Date & Time", "ID"], axis=1, inplace=True)
mcslc_spd["is_cahoots"] = False
springfield = (pd.concat([spd_clean, mcslc_spd], ignore_index=True).sort_values(by="calltime"))
#springfield.drop(["inci id", "Call Year"], axis=1, inplace=True)
springfield.head(3)


,agency,prime unit,closed_as,calltime,yr,month,is_cahoots
0,SPD,1S12,ASSISTED,2015-01-01 00:01:00,2015.0,1.0,False
1,SPD,,DISREGARD,2015-01-01 00:04:00,2015.0,1.0,False
2,SPD,3S12,ARREST,2015-01-01 00:11:00,2015.0,1.0,False


## Download the clean dataframes Eugene and Springfield

In [414]:
eugene.to_csv("eugene.csv", index=False)
springfield.to_csv("springfield.csv", index=False)